# 04 — Feature Engineering
Student Success Analytics Platform

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import json
import pandas as pd
from src.database import run_query
from src.preprocessing import build_feature_matrix, get_column_types
from src.features import compute_vif, get_rf_importances

print("libraries loaded")

libraries loaded


In [2]:
# reload raw X to recover numeric_cols / categorical_cols — mirrors notebook 3 exactly, so column
# groupings stay consistent without needing to persist them separately
df = run_query("SELECT * FROM students;")
X, y_reg, y_clf = build_feature_matrix(df)
numeric_cols, categorical_cols = get_column_types(X)
print(f"numeric: {len(numeric_cols)}  categorical: {len(categorical_cols)}")

numeric: 20  categorical: 11


In [3]:
# load the processed (encoded + scaled) splits from notebook 3 — using the UNRESAMPLED training set.
# feature selection decisions should be based on real data, not synthetic SMOTE samples.
X_train_processed = pd.read_csv("../data/processed/X_train_processed.csv")
X_test_processed = pd.read_csv("../data/processed/X_test_processed.csv")
y_reg_train = pd.read_csv("../data/processed/y_reg_train.csv").squeeze()
y_reg_test = pd.read_csv("../data/processed/y_reg_test.csv").squeeze()
y_clf_train = pd.read_csv("../data/processed/y_clf_train.csv").squeeze()
y_clf_test = pd.read_csv("../data/processed/y_clf_test.csv").squeeze()

print("train:", X_train_processed.shape, " test:", X_test_processed.shape)

train: (64000, 58)  test: (16000, 58)


## Multicollinearity check — engineered composites vs their raw components
`wellness_score` is a linear combination of `mental_health_rating`, `sleep_hours`, `stress_level`, `exercise_frequency`. `distraction_hours` is the literal sum of `social_media_hours` + `netflix_hours`. Both are collinear with their own ingredients **by construction**, not by coincidence — worth confirming how strong that collinearity actually is before deciding what to keep.

In [4]:
collinear_groups = {
    "wellness_score": ["mental_health_rating", "sleep_hours", "stress_level", "exercise_frequency"],
    "distraction_hours": ["social_media_hours", "netflix_hours"],
}

for composite, components in collinear_groups.items():
    print(f"\n{composite} vs components:")
    corrs = X_train_processed[[composite] + components].corr()[composite].drop(composite)
    print(corrs.round(3))


wellness_score vs components:
mental_health_rating    0.473
sleep_hours             0.275
stress_level           -0.476
exercise_frequency      0.719
Name: wellness_score, dtype: float64

distraction_hours vs components:
social_media_hours    0.778
netflix_hours         0.621
Name: distraction_hours, dtype: float64


## Variance Inflation Factor (VIF)
VIF > 10 is the conventional threshold for problematic multicollinearity. Tree-based models (Random Forest, XGBoost) are largely robust to this, but it matters if you ever fit a linear model for interpretability or as a baseline.

In [5]:
vif_table = compute_vif(X_train_processed[numeric_cols])
vif_table

C:\Users\lanre\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,feature,VIF
0,stress_level,inf
1,social_media_hours,inf
2,netflix_hours,inf
3,distraction_hours,inf
4,sleep_hours,inf
5,exercise_frequency,inf
6,mental_health_rating,inf
7,wellness_score,inf
8,screen_time,23.007690
9,study_hours_per_day,13.350434


## Feature importance preview — Random Forest, both targets
Quick, unTuned RF — this is a diagnostic pass for feature selection, not the final model.

In [6]:
importances_reg = get_rf_importances(X_train_processed, y_reg_train, task="regression")
print("top 15 — exam_score:")
print(importances_reg.head(15).round(4))

top 15 — exam_score:
previous_gpa              0.8757
attendance_percentage     0.0084
wellness_score            0.0078
time_management_score     0.0073
stress_level              0.0070
sleep_hours               0.0068
mental_health_rating      0.0068
study_efficiency          0.0063
screen_time               0.0059
netflix_hours             0.0056
study_hours_per_day       0.0055
social_media_hours        0.0054
distraction_hours         0.0049
age                       0.0044
parental_support_level    0.0039
dtype: float64


In [7]:
importances_clf = get_rf_importances(X_train_processed, y_clf_train, task="classification")
print("top 15 — dropout_risk:")
print(importances_clf.head(15).round(4))

top 15 — dropout_risk:
stress_level             0.5227
motivation_level         0.1814
exam_anxiety_score       0.1190
wellness_score           0.0728
previous_gpa             0.0158
mental_health_rating     0.0118
exercise_frequency       0.0096
sleep_hours              0.0053
attendance_percentage    0.0052
study_efficiency         0.0047
time_management_score    0.0045
screen_time              0.0043
study_hours_per_day      0.0042
distraction_hours        0.0041
netflix_hours            0.0038
dtype: float64


## Decision
Compare each composite's importance against the **average** importance of its own raw components. If the composite is pulling more weight than its ingredients individually, the components are candidates to drop (redundant signal, same information). If a component still carries unique signal the composite doesn't capture, keep both.

**Review the printed comparison below before editing `DROP_COLS`** — don't drop anything based on assumption, only on what these numbers actually show.

In [8]:
avg_importance = (importances_reg + importances_clf.reindex(importances_reg.index).fillna(0)) / 2

for composite, components in collinear_groups.items():
    comp_score = avg_importance.get(composite, float("nan"))
    comp_avg = avg_importance.reindex(components).mean()
    print(f"{composite}: {comp_score:.4f}   vs avg({components}): {comp_avg:.4f}")

wellness_score: 0.0403   vs avg(['mental_health_rating', 'sleep_hours', 'stress_level', 'exercise_frequency']): 0.0716
distraction_hours: 0.0045   vs avg(['social_media_hours', 'netflix_hours']): 0.0046


In [9]:
# populate this AFTER reviewing the VIF table and the comparison above — e.g. DROP_COLS = ["social_media_hours", "netflix_hours"]
DROP_COLS = ["wellness_score", "distraction_hours"]

selected_features = [c for c in X_train_processed.columns if c not in DROP_COLS]

X_train_selected = X_train_processed[selected_features]
X_test_selected = X_test_processed[selected_features]

print(f"features: {X_train_processed.shape[1]} -> {len(selected_features)}  (dropped: {DROP_COLS})")

features: 58 -> 56  (dropped: ['wellness_score', 'distraction_hours'])


In [10]:
X_train_selected.to_csv("../data/processed/X_train_selected.csv", index=False)
X_test_selected.to_csv("../data/processed/X_test_selected.csv", index=False)

with open("../data/processed/selected_features.json", "w") as f:
    json.dump(selected_features, f, indent=2)

print("selected feature set saved to ../data/processed/")

selected feature set saved to ../data/processed/


In [11]:
print("=" * 50)
print("  FEATURE ENGINEERING SUMMARY")
print("=" * 50)
print(f"  features before selection: {X_train_processed.shape[1]}")
print(f"  features after selection:  {len(selected_features)}")
print(f"  dropped: {DROP_COLS if DROP_COLS else '(none — review VIF/importance above)'}")
print("=" * 50)
print()
print("next: 05_training.ipynb")

  FEATURE ENGINEERING SUMMARY
  features before selection: 58
  features after selection:  56
  dropped: ['wellness_score', 'distraction_hours']

next: 05_training.ipynb


In [14]:
from src.features import compute_vif, get_rf_importances

vif_result = compute_vif(X_train_processed[numeric_cols])
print(vif_result)

C:\Users\lanre\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


                   feature        VIF
0             stress_level        inf
1       social_media_hours        inf
2            netflix_hours        inf
3        distraction_hours        inf
4              sleep_hours        inf
5       exercise_frequency        inf
6     mental_health_rating        inf
7           wellness_score        inf
8              screen_time  23.007690
9      study_hours_per_day  13.350434
10        motivation_level   6.330789
11      exam_anxiety_score   6.268948
12        study_efficiency   1.783215
13            previous_gpa   1.211279
14                     age   1.000259
15   time_management_score   1.000198
16  parental_support_level   1.000184
17   attendance_percentage   1.000153
18                semester   1.000147
19         social_activity   1.000143


`wellness_score` and `distraction_hours` were dropped from the model feature set. VIF on the
pre-selection numeric block returns `inf` for both columns (along with their constituent inputs:
`sleep_hours`, `exercise_frequency`, `mental_health_rating`, `stress_level` for `wellness_score`;
`social_media_hours`, `netflix_hours` for `distraction_hours`). This is not a "high but finite" VIF —
it reflects exact linear dependence (R² = 1 when regressed against the remaining numeric columns),
since both composites are constructed as linear combinations of columns already present in the
feature set. Including them alongside their components adds no information and produces numerically
unstable coefficient estimates in any linear model. Dropped via `DROP_COLS` in the cell above.

In [15]:
vif_result_post = compute_vif(X_train_processed[numeric_cols].drop(columns=DROP_COLS))
print(vif_result_post)

                   feature        VIF
0              screen_time  23.007690
1      study_hours_per_day  13.350434
2       social_media_hours   7.520121
3         motivation_level   6.330789
4       exam_anxiety_score   6.268948
5            netflix_hours   5.200886
6         study_efficiency   1.783215
7             previous_gpa   1.211279
8             stress_level   1.034505
9     mental_health_rating   1.014275
10             sleep_hours   1.012172
11      exercise_frequency   1.010452
12                     age   1.000259
13   time_management_score   1.000198
14  parental_support_level   1.000184
15   attendance_percentage   1.000153
16                semester   1.000147
17         social_activity   1.000143


### Feature selection: dropping `wellness_score` and `distraction_hours`

Pre-selection VIF (58 features) returns `inf` for `wellness_score`, `distraction_hours`, and their
constituent inputs (`sleep_hours`, `exercise_frequency`, `mental_health_rating`, `stress_level`,
`social_media_hours`, `netflix_hours`). This isn't ordinary high correlation — VIF = inf means R² = 1
against the other numeric columns, i.e. exact linear dependence. Both composites are constructed as
linear combinations of columns already present in the feature set, so they add zero unique variance
and make coefficient estimates in any linear model numerically unstable.

`DROP_COLS = ['wellness_score', 'distraction_hours']`. Post-drop VIF (56 features) confirms no `inf`
values remain. Max VIF is `screen_time` at 23.0 — elevated but not disqualifying (R² < 1, no exact
dependency), consistent with expected correlation between related screen-time metrics
(`social_media_hours`, `netflix_hours`, `screen_time`).